# Zone Leaders Analysis

Dig into the numbers behind the zone leaders graphics — full player-by-zone tables, leader selection math, and insights for social commentary.

## Objective

Which Bulls players dominate each court zone, and what does the data actually say?

## Data Pull

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parents[1]))

from bulls import data, analysis
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

In [ ]:
# Full season shots
shots_season = data.get_team_shots()
shots_season = analysis.detailed_zones(shots_season)
print(f"Season: {len(shots_season):,} shots, {shots_season['game_id'].nunique()} games")

# Last 10 games
shots_last10 = data.get_team_shots(last_n_games=10)
shots_last10 = analysis.detailed_zones(shots_last10)
print(f"Last 10: {len(shots_last10):,} shots, {shots_last10['game_id'].nunique()} games")

# Current roster — used to filter out traded players
roster = data.get_roster()
roster_ids = set(roster['player_id'].tolist())
print(f"\nCurrent roster: {len(roster)} players")
print(roster['player_name'].tolist())

## Helper: Build Player-by-Zone Table

Shared logic used by both PPG and frequency sections.

In [ ]:
def player_zone_table(shots_df):
    """Build a DataFrame with per-player, per-zone stats."""
    df = shots_df[shots_df['shot_zone'] != 'Backcourt'].copy()
    df['points'] = df.apply(
        lambda r: (3 if r['shot_type'] == '3PT' else 2) if r['shot_made'] else 0,
        axis=1,
    )

    rows = []
    for (pid, pname, zone), g in df.groupby(['player_id', 'player_name', 'shot_zone']):
        if pd.isna(pid) or pd.isna(zone):
            continue
        games = g['game_id'].nunique()
        if games == 0:
            continue
        total_pts = g['points'].sum()
        total_shots = len(g)
        rows.append({
            'player_id': int(pid),
            'player_name': pname,
            'zone': zone,
            'ppg': round(total_pts / games, 2),
            'total_points': int(total_pts),
            'total_shots': total_shots,
            'fga_pg': round(total_shots / games, 1),
            'games': games,
        })

    return pd.DataFrame(rows)


def top_n_per_zone(table, metric, n=3, min_shots=5):
    """Return top-N players per zone by a given metric, with min-shots filter."""
    qualified = table[table['total_shots'] >= min_shots]
    return (
        qualified
        .sort_values(metric, ascending=False)
        .groupby('zone')
        .head(n)
        .sort_values(['zone', metric], ascending=[True, False])
        .reset_index(drop=True)
    )


# All-players tables
season_table = player_zone_table(shots_season)
last10_table = player_zone_table(shots_last10)
print(f"Season table: {len(season_table)} player-zone rows")
print(f"Last 10 table: {len(last10_table)} player-zone rows")

# Current-roster-only tables
season_table_cur = season_table[season_table['player_id'].isin(roster_ids)].copy()
last10_table_cur = last10_table[last10_table['player_id'].isin(roster_ids)].copy()

# Current-roster-only shots (for zone_leaders / zone_leaders_by_frequency)
shots_season_cur = shots_season[shots_season['player_id'].isin(roster_ids)].copy()
shots_last10_cur = shots_last10[shots_last10['player_id'].isin(roster_ids)].copy()

traded = set(season_table['player_name']) - set(season_table_cur['player_name'])
if traded:
    print(f"\nTraded players filtered out: {sorted(traded)}")
print(f"Current roster table: {len(season_table_cur)} player-zone rows")

## PPG Leaders (Season)

Full player-by-zone PPG table, plus the top 3 per zone.

In [ ]:
# Full PPG table — every qualifying player/zone combo
ppg_full = (
    season_table[season_table['total_shots'] >= 5]
    [['player_name', 'zone', 'ppg', 'total_points', 'total_shots', 'games']]
    .sort_values(['zone', 'ppg'], ascending=[True, False])
    .reset_index(drop=True)
)
print(f"{len(ppg_full)} qualifying player-zone entries\n")
ppg_full

In [ ]:
# PPG leader per zone (matches the graphic)
ppg_leaders = analysis.zone_leaders(shots_season, min_shots=5)
ppg_leaders_df = pd.DataFrame(ppg_leaders).T
ppg_leaders_df.index.name = 'zone'
print("PPG Leader per Zone (used in graphic):")
ppg_leaders_df[['player_name', 'ppg', 'total_points', 'total_shots', 'games']]

In [ ]:
# Top 3 per zone — see how close runners-up are
ppg_top3 = top_n_per_zone(season_table, 'ppg', n=3, min_shots=5)
for zone in sorted(ppg_top3['zone'].unique()):
    subset = ppg_top3[ppg_top3['zone'] == zone]
    print(f"\n--- {zone} ---")
    print(subset[['player_name', 'ppg', 'total_points', 'total_shots', 'games']].to_string(index=False))

### PPG Leaders — Current Roster Only

In [ ]:
# PPG leaders — current roster only
ppg_leaders_cur = analysis.zone_leaders(shots_season_cur, min_shots=5)
ppg_leaders_cur_df = pd.DataFrame(ppg_leaders_cur).T
ppg_leaders_cur_df.index.name = 'zone'
print("PPG Leader per Zone (current roster only):")
print(ppg_leaders_cur_df[['player_name', 'ppg', 'total_points', 'total_shots', 'games']].to_string())

# Flag zones where the leader changed after removing traded players
print("\n--- Leader changes after filtering traded players ---")
for zone in sorted(set(ppg_leaders.keys()) | set(ppg_leaders_cur.keys())):
    all_name = ppg_leaders.get(zone, {}).get('player_name', '—')
    cur_name = ppg_leaders_cur.get(zone, {}).get('player_name', '—')
    if all_name != cur_name:
        print(f"  {zone}: {all_name} -> {cur_name}")

## Frequency Leaders (Season)

Who takes the most shots per game in each zone?

In [ ]:
# Full frequency table
freq_full = (
    season_table[season_table['total_shots'] >= 5]
    [['player_name', 'zone', 'fga_pg', 'total_shots', 'games']]
    .sort_values(['zone', 'fga_pg'], ascending=[True, False])
    .reset_index(drop=True)
)
print(f"{len(freq_full)} qualifying player-zone entries\n")
freq_full

In [ ]:
# Frequency leader per zone (matches the graphic)
freq_leaders = analysis.zone_leaders_by_frequency(shots_season, min_shots=5)
freq_leaders_df = pd.DataFrame(freq_leaders).T
freq_leaders_df.index.name = 'zone'
print("Frequency Leader per Zone (used in graphic):")
freq_leaders_df[['player_name', 'fga_pg', 'total_shots', 'games']]

In [ ]:
# Top 3 per zone by frequency
freq_top3 = top_n_per_zone(season_table, 'fga_pg', n=3, min_shots=5)
for zone in sorted(freq_top3['zone'].unique()):
    subset = freq_top3[freq_top3['zone'] == zone]
    print(f"\n--- {zone} ---")
    print(subset[['player_name', 'fga_pg', 'total_shots', 'games']].to_string(index=False))

### Frequency Leaders — Current Roster Only

In [ ]:
# Frequency leaders — current roster only
freq_leaders_cur = analysis.zone_leaders_by_frequency(shots_season_cur, min_shots=5)
freq_leaders_cur_df = pd.DataFrame(freq_leaders_cur).T
freq_leaders_cur_df.index.name = 'zone'
print("Frequency Leader per Zone (current roster only):")
print(freq_leaders_cur_df[['player_name', 'fga_pg', 'total_shots', 'games']].to_string())

# Flag zones where the leader changed after removing traded players
print("\n--- Leader changes after filtering traded players ---")
for zone in sorted(set(freq_leaders.keys()) | set(freq_leaders_cur.keys())):
    all_name = freq_leaders.get(zone, {}).get('player_name', '—')
    cur_name = freq_leaders_cur.get(zone, {}).get('player_name', '—')
    if all_name != cur_name:
        print(f"  {zone}: {all_name} -> {cur_name}")

## Last 10 Games Comparison

Same tables for the last 10 games — spot which leaders changed.

In [ ]:
# Last 10: PPG leaders
ppg_leaders_l10 = analysis.zone_leaders(shots_last10, min_shots=3)
ppg_l10_df = pd.DataFrame(ppg_leaders_l10).T
ppg_l10_df.index.name = 'zone'
print("PPG Leaders — Last 10 Games:")
ppg_l10_df[['player_name', 'ppg', 'total_points', 'total_shots', 'games']]

In [ ]:
# Last 10: Frequency leaders
freq_leaders_l10 = analysis.zone_leaders_by_frequency(shots_last10, min_shots=3)
freq_l10_df = pd.DataFrame(freq_leaders_l10).T
freq_l10_df.index.name = 'zone'
print("Frequency Leaders — Last 10 Games:")
freq_l10_df[['player_name', 'fga_pg', 'total_shots', 'games']]

In [ ]:
# Compare: which PPG leaders changed?
print("=== PPG Leader Changes (Season -> Last 10) ===\n")
all_zones = set(ppg_leaders.keys()) | set(ppg_leaders_l10.keys())
for zone in sorted(all_zones):
    season_name = ppg_leaders.get(zone, {}).get('player_name', '—')
    l10_name = ppg_leaders_l10.get(zone, {}).get('player_name', '—')
    marker = ' *CHANGED*' if season_name != l10_name else ''
    print(f"{zone}: {season_name} -> {l10_name}{marker}")

print("\n=== Frequency Leader Changes (Season -> Last 10) ===\n")
all_zones_f = set(freq_leaders.keys()) | set(freq_leaders_l10.keys())
for zone in sorted(all_zones_f):
    season_name = freq_leaders.get(zone, {}).get('player_name', '—')
    l10_name = freq_leaders_l10.get(zone, {}).get('player_name', '—')
    marker = ' *CHANGED*' if season_name != l10_name else ''
    print(f"{zone}: {season_name} -> {l10_name}{marker}")

## Insights

In [ ]:
# 1. Zone dominance — who owns the most zones?
from collections import Counter

print("=== Zone Dominance — All Players (Season) ===\n")
ppg_counts = Counter(v['player_name'] for v in ppg_leaders.values())
freq_counts = Counter(v['player_name'] for v in freq_leaders.values())
print("PPG leader in most zones:")
for name, cnt in ppg_counts.most_common():
    print(f"  {name}: {cnt} zone(s)")
print("\nFrequency leader in most zones:")
for name, cnt in freq_counts.most_common():
    print(f"  {name}: {cnt} zone(s)")

print("\n=== Zone Dominance — Current Roster Only ===\n")
ppg_counts_cur = Counter(v['player_name'] for v in ppg_leaders_cur.values())
freq_counts_cur = Counter(v['player_name'] for v in freq_leaders_cur.values())
print("PPG leader in most zones:")
for name, cnt in ppg_counts_cur.most_common():
    print(f"  {name}: {cnt} zone(s)")
print("\nFrequency leader in most zones:")
for name, cnt in freq_counts_cur.most_common():
    print(f"  {name}: {cnt} zone(s)")

In [ ]:
# 2. High-volume / low-efficiency (and vice versa)
# Merge PPG and frequency rankings per zone
merged = season_table[season_table['total_shots'] >= 5].copy()

# Per-zone percentile ranks
merged['ppg_rank'] = merged.groupby('zone')['ppg'].rank(ascending=False)
merged['fga_rank'] = merged.groupby('zone')['fga_pg'].rank(ascending=False)

# High volume (top 3 FGA) but low efficiency (not top 3 PPG)
high_vol_low_eff = merged[(merged['fga_rank'] <= 3) & (merged['ppg_rank'] > 3)]
print("=== High Volume, Lower Efficiency ===\n")
if high_vol_low_eff.empty:
    print("None — top shooters by volume are also top by PPG in their zones.")
else:
    print(high_vol_low_eff[['player_name', 'zone', 'fga_pg', 'ppg', 'total_shots']].to_string(index=False))

# Low volume (not top 3 FGA) but high efficiency (top 3 PPG)
low_vol_high_eff = merged[(merged['ppg_rank'] <= 3) & (merged['fga_rank'] > 3)]
print("\n=== Low Volume, High Efficiency ===\n")
if low_vol_high_eff.empty:
    print("None — top PPG players also take the most shots in their zones.")
else:
    print(low_vol_high_eff[['player_name', 'zone', 'ppg', 'fga_pg', 'total_shots']].to_string(index=False))

In [ ]:
# 3. Zones where PPG leader != Frequency leader
print("=== PPG vs Frequency Leader Mismatches ===\n")
shared_zones = set(ppg_leaders.keys()) & set(freq_leaders.keys())
mismatches = []
for zone in sorted(shared_zones):
    ppg_name = ppg_leaders[zone]['player_name']
    freq_name = freq_leaders[zone]['player_name']
    if ppg_name != freq_name:
        mismatches.append({
            'zone': zone,
            'ppg_leader': ppg_name,
            'ppg': ppg_leaders[zone]['ppg'],
            'freq_leader': freq_name,
            'fga_pg': freq_leaders[zone]['fga_pg'],
        })

if mismatches:
    mismatch_df = pd.DataFrame(mismatches)
    print(mismatch_df.to_string(index=False))
    print(f"\n{len(mismatches)} of {len(shared_zones)} zones have different PPG vs frequency leaders.")
else:
    print("Every zone's PPG leader is also its frequency leader.")

## Takeaways

- *Fill after running: Who dominates the most zones (PPG and frequency)?*
- *Fill after running: Which zones have a PPG/frequency mismatch — efficiency vs. volume story?*
- *Fill after running: What changed in the last 10 games vs. full season?*
- *Fill after running: Any low-volume high-efficiency players worth featuring?*

## Next Question

How do the Bulls' zone leaders compare against league-average PPS in each zone?